# Comparing `semgrep scan` vs `semgrep ci`

**Purpose:** Compare what `semgrep scan` (local CLI) and `semgrep ci` (cloud-augmented) produce
when run against the same repository. Understand what the cloud tier adds — and what you lose
if you only use the local scanner.

This notebook runs `semgrep scan` against the DevSecOps-Kit repo, then explains the
additional capabilities `semgrep ci` would bring.

> Requires: `semgrep` installed (`pip install semgrep`), and a Semgrep App account for `semgrep ci`.

In [ ]:
import subprocess
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).resolve()
SCAN_TARGET = str(REPO_ROOT)
print(f"Scan target: {SCAN_TARGET}")

In [ ]:
# Verify semgrep is installed
result = subprocess.run(["semgrep", "--version"], capture_output=True, text=True)
if result.returncode != 0:
    print("semgrep not found. Install with: pip install semgrep")
    sys.exit(1)
print(f"Semgrep version: {result.stdout.strip()}")

---
## Scan 1: `semgrep scan` (local CLI)

The local CLI runs all matching rules from the Semgrep Registry against the target directory.
Results are returned immediately with no cloud augmentation — no triage, no PR comments,
no findings deduplication across branches.

In [ ]:
print("Running semgrep scan --config=auto ... (this may take a minute)\n")
scan_result = subprocess.run(
    ["semgrep", "scan", "--config=auto", "--json", "--quiet",
     "--exclude", "node_modules", "--exclude", ".git",
     "--exclude", "__pycache__", "--exclude", "venv",
     "--max-target-bytes", "50000",
     SCAN_TARGET],
    capture_output=True, text=True, timeout=120
)

if scan_result.returncode not in (0, 1):
    print(f"semgrep scan failed (exit {scan_result.returncode})")
    print(scan_result.stderr[:500])
else:
    print(f"Exit code: {scan_result.returncode} (0 = clean, 1 = findings)")

In [ ]:
data = json.loads(scan_result.stdout)
results = data.get("results", [])
errors = data.get("errors", [])

print(f"Rules run: {len(data.get('paths', {}).get('scanned', []))} files scanned")
print(f"Total findings: {len(results)}")
print(f"Errors: {len(errors)}")

# Group findings by severity
severity_counts = {}
for r in results:
    sev = r.get("extra", {}).get("severity", "unknown")
    severity_counts[sev] = severity_counts.get(sev, 0) + 1

print("\nFindings by severity:")
for sev in ["ERROR", "WARNING", "INFO"]:
    count = severity_counts.get(sev, 0)
    print(f"  {sev}: {count}")


# Show a few sample findings
print("\n--- Sample findings (first 3) ---")
for r in results[:3]:
    path = r.get("path", "?")
    line = r.get("start", {}).get("line", "?")
    msg = r.get("extra", {}).get("message", "")[:120]
    rule = r.get("check_id", "?")
    print(f"  [{r.get('extra',{}).get('severity','?')}] {path}:{line}")
    print(f"         Rule: {rule}")
    print(f"         {msg}\n")

---
## Scan 2: `semgrep ci` (cloud-augmented)

`semgrep ci` wraps the local scan with cloud services:

- **Findings triage** — Semgrep App can suppress known-wont-fix findings across runs
- **PR/MR comments** — posts findings as comments on pull requests
- **Branch tracking** — tracks findings per-branch to detect new vs existing issues
- **Rule tuning** — the cloud API adjusts rule confidence based on your project's false-positive patterns
- **Findings dashboard** — web UI for filtering, assigning, and trending

To run this cell, set `SEMGREP_APP_TOKEN` in your environment and uncomment:
```python
# ci_result = subprocess.run(["semgrep", "ci", "--json"], ...)
```

Below, we simulate the cloud augmentation by checking environment variables and showing
what additional metadata `semgrep ci` would provide.

In [ ]:
has_token = "SEMGREP_APP_TOKEN" in os.environ
ci_available = False

if has_token:
    print("SEMGREP_APP_TOKEN found — semgrep ci can authenticate to Semgrep App")
else:
    print("SEMGREP_APP_TOKEN not set — semgrep ci will run in local-only mode")
    print("  Same as semgrep scan, but with --config auto and a different output format")

print("\nWhat semgrep ci adds beyond semgrep scan:")
additions = [
    "Tracks findings per branch — new vs existing separation",
    "PR comments with findings (via GitHub/GitLab/Slack integration)",
    "Findings triage — suppress false positives permanently",
    "Rule tuning based on project-specific feedback",
    "Dashboard with trends, assignees, and filters",
]
for a in additions:
    print(f"  • {a}")

---
## Comparison

| Aspect | `semgrep scan` | `semgrep ci` |
|---|---|---|
| Rules | Local registry + custom | Same + cloud-tuned |
| Speed | Faster (no upload) | Slower (upload logs + findings) |
| Findings | Raw list | Triaged — known vs new |
| CI integration | Manual exit-code gating | Native PR comments + blocking |
| Dashboard | None | Web UI at semgrep.dev |
| Offline | Yes | No (requires API) |

**Takeaway:** Use `semgrep scan` for local dev loops, CI-only safety checks, or air-gapped
environments. Use `semgrep ci` in CI pipelines where cloud access is available and you
need triage, PR feedback, and cross-branch tracking.

In [ ]:
# Summary counts as a quick reference
print("=== Summary ===")
print(f"Files scanned: {len(data.get('paths', {}).get('scanned', []))}")
print(f"Total findings (semgrep scan): {len(results)}")
print(f"  ERROR:   {severity_counts.get('ERROR', 0)}")
print(f"  WARNING: {severity_counts.get('WARNING', 0)}")
print(f"  INFO:    {severity_counts.get('INFO', 0)}")
print(f"\nCloud augmentation (semgrep ci) adds: triage, PR comments, branch tracking")

---
## Verify

To confirm the comparison yourself:

1. Run `semgrep scan --config=auto --json <repo>` and save the JSON
2. If you have a Semgrep App account, run `semgrep ci --json` on the same repo
3. Compare these differences:
   - Does `semgrep ci` show fewer findings? (cloud triage suppresses noise)
   - Does it tag some as `blocking` vs `non-blocking`?
   - Does it reference a finding ID that links back to the dashboard?

If you can't run `semgrep ci`, the conceptual comparison above still holds — the key
difference is the cloud layer, not the scanning engine.